# 02 - churn model benchmark

## Goal

Compare five churn models fairly, select a champion without touching the final test set, and save a calibrated probability model suitable for batch scoring.

## Recommended validation design

For this 7,043-customer dataset, use an **80/20 stratified holdout test split** and **five-fold stratified cross-validation on the training data**. This gives one independent final test set while using the remaining data efficiently during tuning.

Nested cross-validation is more statistically conservative, but is expensive for five tuned models. The chosen approach is practical for this assignment and is appropriate when the final holdout test set is used only once.

## Reproducibility settings

Every random operation uses the same seed. Record these settings in the report so another group member can reproduce the result.

In [8]:
SEED = 42
TEST_SIZE = 0.20
VALIDATION_FOLDS = 5
CALIBRATION_FOLDS = 5
SEARCH_ITERATIONS = 20
BOOTSTRAP_REPEATS = 400
BATCH_SIZE = 1_000

import random
import numpy as np

random.seed(SEED)
np.random.seed(SEED)
print({'seed': SEED, 'test_size': TEST_SIZE, 'validation_folds': VALIDATION_FOLDS})

{'seed': 42, 'test_size': 0.2, 'validation_folds': 5}


## Model comparison: strengths and limitations

The model is selected by validation PR-AUC. Brier score and calibration are then checked before the probability is used for retention decisions.

In [9]:
import pandas as pd

model_notes = pd.DataFrame([
    ['Logistic regression', 'Fast, stable, easy to explain', 'May miss non-linear churn patterns'],
    ['SVC (RBF)', 'Captures smooth non-linear boundaries', 'Slow at large scale; probability fitting costs time'],
    ['Random forest', 'Robust non-linear baseline', 'Can produce poorly calibrated probabilities'],
    ['XGBoost', 'Strong tabular-data performance', 'Needs tuning and monitoring for overfitting'],
    ['PyTorch MLP', 'Learns complex interactions and supports GPU later', 'Sensitive to scaling and tuning'],
], columns=['model', 'advantages', 'limitations'])
display(model_notes)

,model,advantages,limitations
0,Logistic regression,"Fast, stable, easy to explain",May miss non-linear churn patterns
1,SVC (RBF),Captures smooth non-linear boundaries,Slow at large scale; probability fitting costs...
2,Random forest,Robust non-linear baseline,Can produce poorly calibrated probabilities
3,XGBoost,Strong tabular-data performance,Needs tuning and monitoring for overfitting
4,PyTorch MLP,Learns complex interactions and supports GPU l...,Sensitive to scaling and tuning


## Hyperparameter ranges

The ranges below follow the requested benchmark settings. `RandomizedSearchCV` samples 20 candidate configurations for each model using five folds.

In [10]:
search_plan = pd.DataFrame([
    ['Logistic regression', 'C', '1e-4 to 1e4'],
    ['Logistic regression', 'l1_ratio', '0 to 1 (Elastic Net)'],
    ['SVC (RBF)', 'C', '1e2 to 1e3'],
    ['SVC (RBF)', 'gamma', '1e-4 to 1'],
    ['Random forest', 'n_estimators', '300 to 500'],
    ['Random forest', 'max_depth', '5 to 40'],
    ['Random forest', 'min_samples_leaf', '1 to 10'],
    ['XGBoost', 'n_estimators', '200 to 300'],
    ['XGBoost', 'learning_rate', '0.01 to 0.30'],
    ['XGBoost', 'max_depth', '3 to 10'],
    ['XGBoost', 'min_child_weight', '1 to 10'],
    ['PyTorch MLP', 'hidden_layer_sizes', '(128), (256), (128, 64)'],
    ['PyTorch MLP', 'alpha / weight decay', '1e-5 to 1e-2'],
    ['PyTorch MLP', 'learning_rate_init', '1e-4 to 1e-2'],
], columns=['model', 'parameter', 'search range'])
display(search_plan)

,model,parameter,search range
0,Logistic regression,C,1e-4 to 1e4
1,Logistic regression,l1_ratio,0 to 1 (Elastic Net)
2,SVC (RBF),C,1e2 to 1e3
3,SVC (RBF),gamma,1e-4 to 1
4,Random forest,n_estimators,300 to 500
5,Random forest,max_depth,5 to 40
6,Random forest,min_samples_leaf,1 to 10
7,XGBoost,n_estimators,200 to 300
8,XGBoost,learning_rate,0.01 to 0.30
9,XGBoost,max_depth,3 to 10


## Load data and create the final test split

The final test split is stratified, which preserves the churn rate in training and test sets. Do not tune model settings against this test set.

In [11]:
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, train_test_split

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
df = pd.read_csv(ROOT / 'data' / 'telco_clean_32col.csv')
X = df.drop(columns='Churn')
y = df['Churn'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=SEED,
)

validation_cv = StratifiedKFold(
    n_splits=VALIDATION_FOLDS,
    shuffle=True,
    random_state=SEED,
)
calibration_cv = StratifiedKFold(
    n_splits=CALIBRATION_FOLDS,
    shuffle=True,
    random_state=SEED,
)
print('Train rows:', len(X_train), 'Test rows:', len(X_test))
print('Train churn rate:', round(y_train.mean(), 3), 'Test churn rate:', round(y_test.mean(), 3))

Train rows: 5634 Test rows: 1409
Train churn rate: 0.265 Test churn rate: 0.265


## Define five reproducible model pipelines

Scaling is included only for models sensitive to feature magnitude. Tree models receive the original engineered numeric features.

In [12]:
from scipy.stats import loguniform, randint, uniform
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Make the reusable PyTorch estimator importable from notebooks/.
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.torch_mlp import TorchMLPClassifier

models = {
    'Logistic regression': (
        Pipeline([
            ('scale', StandardScaler()),
            ('model', LogisticRegression(
                penalty='elasticnet', solver='saga', max_iter=3000,
                class_weight='balanced', random_state=SEED,
            )),
        ]),
        {'model__C': loguniform(1e-4, 1e4), 'model__l1_ratio': uniform(0, 1)},
    ),
    'SVC (RBF)': (
        Pipeline([
            ('scale', StandardScaler()),
            ('model', SVC(
                kernel='rbf', probability=True, class_weight='balanced',
                random_state=SEED,
            )),
        ]),
        {'model__C': loguniform(1e2, 1e3), 'model__gamma': loguniform(1e-4, 1)},
    ),
    'Random forest': (
        RandomForestClassifier(
            class_weight='balanced', random_state=SEED, n_jobs=-1,
        ),
        {
            'n_estimators': randint(300, 501),
            'max_depth': randint(5, 41),
            'min_samples_leaf': randint(1, 11),
        },
    ),
    'XGBoost': ( 
        XGBClassifier(
            objective='binary:logistic', eval_metric='aucpr',
            tree_method='hist', random_state=SEED, n_jobs=-1,
        ),
        {
            'n_estimators': randint(200, 301),
            'learning_rate': loguniform(.01, .30),
            'max_depth': randint(3, 11),
            'min_child_weight': randint(1, 11),
        },
    ),
    'Neural network (PyTorch MLP)': (
        Pipeline([
            ('scale', StandardScaler()),
            ('model', TorchMLPClassifier(
                batch_size=256, epochs=100, patience=12,
                random_state=SEED, device='cpu',
            )),
        ]),
        {
            'model__hidden_layer_sizes': [(128,), (256,), (128, 64)],
            'model__alpha': loguniform(1e-5, 1e-2),
            'model__learning_rate_init': loguniform(1e-4, 1e-2),
        },
    ),
}
print('Five model pipelines are ready.')

Five model pipelines are ready.


## Tune, calibrate, and evaluate every model

For each model: (1) tune hyperparameters with five-fold PR-AUC, (2) calibrate probabilities with five-fold Platt scaling, and (3) predict the untouched final test set.

The validation mean plus/minus standard deviation reports fold-to-fold stability. Test metrics are reported with bootstrap plus/minus standard deviation to show uncertainty.

In [13]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV
import numpy as np
import pandas as pd


# THRESHOLD = (0.30,0.55, 0.65)
THRESHOLD = 0.30

def bootstrap_standard_deviation(y_true, probability, metric_function):
    """Estimate test-metric uncertainty using deterministic bootstrap samples."""
    rng = np.random.default_rng(SEED)
    y_array = np.asarray(y_true)
    p_array = np.asarray(probability)
    values = []

    for _ in range(BOOTSTRAP_REPEATS):
        sample_index = rng.integers(0, len(y_array), len(y_array))
        sample_y = y_array[sample_index]
        sample_p = p_array[sample_index]

        if len(np.unique(sample_y)) == 2:
            values.append(metric_function(sample_y, sample_p))

    return float(np.std(values, ddof=1))


results = []
fitted_models = {}

for model_name, (estimator, parameter_space) in models.items():
    print(f'Tuning {model_name}...')

    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=parameter_space,
        n_iter=SEARCH_ITERATIONS,
        scoring='average_precision',
        cv=validation_cv,
        random_state=SEED,
        n_jobs=-1,
        refit=True,
    )

    search.fit(X_train, y_train)

    # Calibrate the selected model on training folds only.
    calibrated_model = CalibratedClassifierCV(
        estimator=search.best_estimator_,
        method='sigmoid',
        cv=calibration_cv,
        n_jobs=-1,
    )

    calibrated_model.fit(X_train, y_train)

    test_probability = calibrated_model.predict_proba(X_test)[:, 1]

    # Use the chosen retention threshold.
    test_prediction = (test_probability >= THRESHOLD).astype(int)

    cv_mean = search.cv_results_['mean_test_score'][search.best_index_]
    cv_std = search.cv_results_['std_test_score'][search.best_index_]

    test_accuracy = accuracy_score(y_test, test_prediction)
    test_balanced_accuracy = balanced_accuracy_score(y_test, test_prediction)
    test_predicted_churn_rate = float(test_prediction.mean())

    results.append({
        'model': model_name,
        'threshold': THRESHOLD,
        'validation_pr_auc_mean': cv_mean,
        'validation_pr_auc_std': cv_std,
        'test_roc_auc': roc_auc_score(y_test, test_probability),
        'test_pr_auc': average_precision_score(y_test, test_probability),
        'test_brier_score': brier_score_loss(y_test, test_probability),

        # Correct threshold-based accuracy columns.
        'test_accuracy_at_threshold': test_accuracy,
        'test_balanced_accuracy_at_threshold': test_balanced_accuracy,
        'test_predicted_churn_rate_at_threshold': test_predicted_churn_rate,

        'test_roc_auc_std': bootstrap_standard_deviation(
            y_test,
            test_probability,
            roc_auc_score,
        ),
        'test_pr_auc_std': bootstrap_standard_deviation(
            y_test,
            test_probability,
            average_precision_score,
        ),
        'test_brier_score_std': bootstrap_standard_deviation(
            y_test,
            test_probability,
            brier_score_loss,
        ),
        'best_parameters': search.best_params_,
    })

    fitted_models[model_name] = calibrated_model

    print(
        f'{model_name}: '
        f'Threshold={THRESHOLD:.2f}, '
        f'Accuracy={test_accuracy:.3f}, '
        f'Balanced accuracy={test_balanced_accuracy:.3f}, '
        f'Predicted churn rate={test_predicted_churn_rate:.3f}'
    )


results_table = pd.DataFrame(results).sort_values(
    'validation_pr_auc_mean',
    ascending=False,
).reset_index(drop=True)

display(
    results_table[[
        'model',
        'threshold',
        'validation_pr_auc_mean',
        'validation_pr_auc_std',
        'test_roc_auc',
        'test_pr_auc',
        'test_brier_score',
        'test_accuracy_at_threshold',
        'test_balanced_accuracy_at_threshold',
        'test_predicted_churn_rate_at_threshold',
    ]]
)

display(results_table)

Tuning Logistic regression...


KeyboardInterrupt: 

Tuning Logistic regression...


c:\Users\mikil\Saved Games\new\env\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Tuning SVC (RBF)...


c:\Users\mikil\Saved Games\new\env\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Tuning Random forest...
Tuning XGBoost...
Tuning Neural network (PyTorch MLP)...


,model,validation_pr_auc_mean,validation_pr_auc_std,test_roc_auc,test_pr_auc,test_brier_score,test_accuracy_at_0_50,test_balanced_accuracy_at_0_50,test_roc_auc_std,test_pr_auc_std,test_brier_score_std,best_parameters
0,SVC (RBF),0.670413,0.018999,0.845545,0.667368,0.136607,0.792761,0.719767,0.011094,0.024651,0.005314,"{'model__C': 201.65721691808585, 'model__gamma..."
1,Logistic regression,0.670391,0.016491,0.847640,0.664698,0.134617,0.807665,0.720520,0.011088,0.025220,0.005357,"{'model__C': 0.09915644566638401, 'model__l1_r..."
2,Neural network (PyTorch MLP),0.668244,0.016207,0.846258,0.653065,0.137077,0.806246,0.715285,0.011207,0.026189,0.005935,"{'model__alpha': 0.008123245085588688, 'model_..."
3,XGBoost,0.666050,0.021042,0.844005,0.657046,0.137619,0.799148,0.704477,0.011280,0.025160,0.005766,"{'learning_rate': 0.04616803492122799, 'max_de..."
4,Random forest,0.664804,0.020494,0.845083,0.654864,0.136254,0.801278,0.728979,0.011485,0.025776,0.005565,"{'max_depth': 7, 'min_samples_leaf': 6, 'n_est..."


## Read the results as mean plus/minus uncertainty

Choose the champion from validation performance only. The final test table is for one-time confirmation. A high PR-AUC is useful only if Brier score and calibration are also acceptable.

In [ ]:
summary_table = results_table.assign(
    validation_pr_auc=lambda table: table.apply(
        lambda row: f"{row['validation_pr_auc_mean']:.3f} +/- {row['validation_pr_auc_std']:.3f}",
        axis=1,
    ),
    test_roc_auc=lambda table: table.apply(
        lambda row: f"{row['test_roc_auc']:.3f} +/- {row['test_roc_auc_std']:.3f}",
        axis=1,
    ),
    test_pr_auc=lambda table: table.apply(
        lambda row: f"{row['test_pr_auc']:.3f} +/- {row['test_pr_auc_std']:.3f}",
        axis=1,
    ),
    test_brier=lambda table: table.apply(
        lambda row: f"{row['test_brier_score']:.3f} +/- {row['test_brier_score_std']:.3f}",
        axis=1,
    ),
    # Format the new threshold-based metrics
    test_accuracy=lambda table: table['test_accuracy_at_threshold'].apply(lambda x: f"{x:.3f}"),
    test_balanced_accuracy=lambda table: table['test_balanced_accuracy_at_threshold'].apply(lambda x: f"{x:.3f}"),
    predicted_churn_rate=lambda table: table['test_predicted_churn_rate_at_threshold'].apply(lambda x: f"{x:.3f}"),
)

display(summary_table[[
    'model', 
    'validation_pr_auc', 
    'test_roc_auc', 
    'test_pr_auc', 
    'test_brier',
    'test_accuracy',
    'test_balanced_accuracy',
    'predicted_churn_rate',
]])

champion_name = results_table.iloc[0]['model']
champion_model = fitted_models[champion_name]
print('Champion selected by validation PR-AUC:', champion_name)

,model,validation_pr_auc,test_roc_auc,test_pr_auc,test_brier,test_accuracy,test_balanced_accuracy,predicted_churn_rate
0,SVC (RBF),0.670 +/- 0.019,0.846 +/- 0.011,0.667 +/- 0.025,0.137 +/- 0.005,0.755,0.759,0.387
1,Logistic regression,0.670 +/- 0.016,0.848 +/- 0.011,0.665 +/- 0.025,0.135 +/- 0.005,0.760,0.763,0.383
2,Neural network (PyTorch MLP),0.668 +/- 0.016,0.846 +/- 0.011,0.653 +/- 0.026,0.137 +/- 0.006,0.779,0.757,0.334
3,XGBoost,0.666 +/- 0.021,0.844 +/- 0.011,0.657 +/- 0.025,0.138 +/- 0.006,0.774,0.753,0.336
4,Random forest,0.665 +/- 0.020,0.845 +/- 0.011,0.655 +/- 0.026,0.136 +/- 0.006,0.768,0.769,0.375


Champion selected by validation PR-AUC: SVC (RBF)


## Save the production artifact and batch-score customers

This dataset is small, so training does not need mini-batches. In production, score customers in batches to control memory use and schedule regular refreshes. The input schema must match the engineered feature columns used during training.

In [ ]:
import json
import joblib

ARTIFACTS = ROOT / 'artifacts'
ARTIFACTS.mkdir(exist_ok=True)

joblib.dump(champion_model, ARTIFACTS / 'production_churn_model.joblib')
results_table.to_csv(ARTIFACTS / 'benchmark_summary.csv', index=False)

# Shared evaluation bundle for calibration, SHAP, and retention notebooks.
joblib.dump(
    {'model': champion_model, 'X_test': X_test, 'y_test': y_test},
    ARTIFACTS / 'model.joblib',
)
joblib.dump(
    {
        'models': fitted_models,
        'results': results_table,
        'X_test': X_test,
        'y_test': y_test,
    },
    ARTIFACTS / 'benchmark_models.pkl',
)

metadata = {
    'seed': SEED,
    'test_size': TEST_SIZE,
    'validation_folds': VALIDATION_FOLDS,
    'calibration_folds': CALIBRATION_FOLDS,
    'champion_model': champion_name,
    'feature_columns': X.columns.tolist(),
}
with open(ARTIFACTS / 'production_metadata.json', 'w', encoding='utf-8') as handle:
    json.dump(metadata, handle, indent=2)

def score_in_batches(customer_features, fitted_model, batch_size=BATCH_SIZE):
    """Return calibrated churn probabilities without loading all rows at once."""
    scores = []
    for start in range(0, len(customer_features), batch_size):
        batch = customer_features.iloc[start:start + batch_size]
        scores.append(fitted_model.predict_proba(batch)[:, 1])
    return np.concatenate(scores)

# Example production-style batch scoring on the held-out customers.
batch_scores = score_in_batches(X_test, champion_model)
print('Saved production model. Batch-scored customers:', len(batch_scores))

Saved production model. Batch-scored customers: 1409


## What to write in the report

State the seed, 80/20 stratified split, five validation folds, calibration method, search ranges, and selection rule. Present every model as validation PR-AUC mean +/- standard deviation and final test metrics +/- bootstrap standard deviation. Then justify the champion by its ranking quality, Brier score, and calibration curve rather than accuracy alone.

## Save the champion to Databricks MLflow

Run this cell in Databricks after training.

In [ ]:
import numpy as np
import pandas as pd
import mlflow
from mlflow.models import infer_signature
from mlflow import MlflowClient
from sklearn.model_selection import RandomizedSearchCV
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, confusion_matrix, f1_score, log_loss,
    precision_recall_curve, precision_score, recall_score,
    roc_auc_score, roc_curve,
)
import matplotlib.pyplot as plt

# ---------------- Config ----------------
SEED = 42
THRESHOLD = 0.30                 # single value — see the earlier fix
VALIDATION_FOLDS = 5
CALIBRATION_FOLDS = 5
SEARCH_ITERATIONS = 20
BOOTSTRAP_REPEATS = 400

MODEL = 'prod.tccp-cicd.TCCP'
mlflow.set_tracking_uri('databricks')
mlflow.set_registry_uri('databricks-uc')
mlflow.set_experiment(experiment_id='1783740843977330')


def bootstrap_standard_deviation(y_true, probability, metric_function):
    rng = np.random.default_rng(SEED)
    y_array, p_array = np.asarray(y_true), np.asarray(probability)
    values = []
    for _ in range(BOOTSTRAP_REPEATS):
        idx = rng.integers(0, len(y_array), len(y_array))
        sy, sp = y_array[idx], p_array[idx]
        if len(np.unique(sy)) == 2:
            values.append(metric_function(sy, sp))
    return float(np.std(values, ddof=1))


def full_metrics(y_test, probability, threshold=THRESHOLD):
    prediction = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, prediction).ravel()
    return {
        'test_roc_auc': roc_auc_score(y_test, probability),
        'test_pr_auc': average_precision_score(y_test, probability),
        'test_brier_score': brier_score_loss(y_test, probability),
        'test_log_loss': log_loss(y_test, probability),
        'test_accuracy': accuracy_score(y_test, prediction),
        'test_balanced_accuracy': balanced_accuracy_score(y_test, prediction),
        'test_precision': precision_score(y_test, prediction, zero_division=0),
        'test_recall': recall_score(y_test, prediction, zero_division=0),
        'test_f1': f1_score(y_test, prediction, zero_division=0),
        'test_specificity': tn / (tn + fp) if (tn + fp) else 0.0,
        'test_negative_predictive_value': tn / (tn + fn) if (tn + fn) else 0.0,
        'test_flagged_rate': float(prediction.mean()),
        'test_true_positives': float(tp), 'test_false_positives': float(fp),
        'test_true_negatives': float(tn), 'test_false_negatives': float(fn),
    }, prediction


results = []
fitted_models = {}

with mlflow.start_run(run_name='TCCP_benchmark') as parent_run:
    mlflow.log_params({
        'decision_threshold': THRESHOLD,
        'search_iterations': SEARCH_ITERATIONS,
        'validation_folds': VALIDATION_FOLDS,
        'calibration_folds': CALIBRATION_FOLDS,
        'n_candidate_models': len(models),
    })

    for model_name, (estimator, parameter_space) in models.items():
        print(f'Tuning {model_name}...')

        with mlflow.start_run(run_name=model_name, nested=True):
            search = RandomizedSearchCV(
                estimator=estimator, param_distributions=parameter_space,
                n_iter=SEARCH_ITERATIONS, scoring='average_precision',
                cv=validation_cv, random_state=SEED, n_jobs=-1, refit=True,
            )
            search.fit(X_train, y_train)

            calibrated_model = CalibratedClassifierCV(
                estimator=search.best_estimator_, method='sigmoid',
                cv=calibration_cv, n_jobs=-1,
            )
            calibrated_model.fit(X_train, y_train)

            test_probability = calibrated_model.predict_proba(X_test)[:, 1]
            metrics, test_prediction = full_metrics(y_test, test_probability)

            cv_mean = search.cv_results_['mean_test_score'][search.best_index_]
            cv_std = search.cv_results_['std_test_score'][search.best_index_]

            mlflow.log_params({
                'model': model_name,
                'decision_threshold': THRESHOLD,
                **{f'best_{k}': v for k, v in search.best_params_.items()},
            })
            mlflow.log_metrics({
                'validation_pr_auc_mean': cv_mean,
                'validation_pr_auc_std': cv_std,
                'test_roc_auc_std': bootstrap_standard_deviation(y_test, test_probability, roc_auc_score),
                'test_pr_auc_std': bootstrap_standard_deviation(y_test, test_probability, average_precision_score),
                'test_brier_score_std': bootstrap_standard_deviation(y_test, test_probability, brier_score_loss),
                **metrics,
            })

            results.append({
                'model': model_name, 'threshold': THRESHOLD,
                'validation_pr_auc_mean': cv_mean, 'validation_pr_auc_std': cv_std,
                **metrics, 'best_parameters': search.best_params_,
            })
            fitted_models[model_name] = calibrated_model

            print(f'{model_name}: Accuracy={metrics["test_accuracy"]:.3f}, '
                  f'Recall={metrics["test_recall"]:.3f}, PR-AUC={cv_mean:.3f}')

    results_table = pd.DataFrame(results).sort_values('validation_pr_auc_mean', ascending=False).reset_index(drop=True)
    champion_name = results_table.iloc[0]['model']
    champion_model = fitted_models[champion_name]
    print(f'\nChampion selected by validation PR-AUC: {champion_name}')

    mlflow.log_text(results_table.drop(columns='best_parameters').to_csv(index=False),
                     'evaluation/benchmark_summary_all_models.csv')

    # ---- Full evaluation + registration for the champion only (your original block) ----
    model_input_example = X_test.head(5)
    model_signature = infer_signature(model_input_example, champion_model.predict(model_input_example))

    champion_probability = champion_model.predict_proba(X_test)[:, 1]
    champion_prediction = (champion_probability >= THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, champion_prediction).ravel()

    mlflow.log_params({'champion_model': champion_name, 'decision_threshold': float(THRESHOLD)})
    mlflow.log_metrics(full_metrics(y_test, champion_probability)[0])

    fpr, tpr, _ = roc_curve(y_test, champion_probability)
    precision, recall, _ = precision_recall_curve(y_test, champion_probability)
    calibration_pred, calibration_obs = calibration_curve(y_test, champion_probability, n_bins=10, strategy='quantile')
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    axes[0, 0].plot(fpr, tpr, label=f'ROC-AUC = {roc_auc_score(y_test, champion_probability):.3f}', color='#264653', linewidth=2)
    axes[0, 0].plot([0, 1], [0, 1], '--', color='grey')
    axes[0, 0].set(title='Ranking performance: ROC curve', xlabel='False-positive rate', ylabel='True-positive rate')
    axes[0, 0].legend(frameon=False)
    axes[0, 1].plot(recall, precision, label=f'PR-AUC = {average_precision_score(y_test, champion_probability):.3f}', color='#d1495b', linewidth=2)
    axes[0, 1].axhline(np.mean(y_test), linestyle='--', color='grey', label=f'Churn prevalence = {np.mean(y_test):.3f}')
    axes[0, 1].set(title='Churn identification: PR curve', xlabel='Recall', ylabel='Precision')
    axes[0, 1].legend(frameon=False)
    axes[1, 0].plot(calibration_pred, calibration_obs, 'o-', color='#2a9d8f', label='Champion model')
    axes[1, 0].plot([0, 1], [0, 1], '--', color='grey', label='Perfect calibration')
    axes[1, 0].set(title=f'Probability reliability: Brier = {brier_score_loss(y_test, champion_probability):.3f}', xlabel='Mean predicted risk', ylabel='Observed churn rate', xlim=(0, 1), ylim=(0, 1))
    axes[1, 0].legend(frameon=False)
    matrix = np.array([[tn, fp], [fn, tp]])
    axes[1, 1].imshow(matrix, cmap='Blues')
    axes[1, 1].set(title=f'Confusion matrix at threshold {THRESHOLD:.2f}', xticks=[0, 1], yticks=[0, 1],
                    xticklabels=['Predicted stay', 'Predicted churn'], yticklabels=['Actual stay', 'Actual churn'])
    for (row, col), value in np.ndenumerate(matrix):
        axes[1, 1].text(col, row, int(value), ha='center', va='center',
                         color='white' if value > matrix.max() / 2 else 'black', fontsize=13)
    fig.suptitle(f'TCCP champion evaluation | {champion_name}', fontsize=15, fontweight='bold')
    fig.tight_layout()
    mlflow.log_figure(fig, 'evaluation/tccp_evaluation_dashboard.png')
    plt.close(fig)

    mlflow.sklearn.log_model(
        champion_model, name='model', registered_model_name=MODEL,
        input_example=model_input_example, signature=model_signature,
        skops_trusted_types=[
            'sklearn.calibration._CalibratedClassifier',
            'sklearn.calibration._SigmoidCalibration',
            'sklearn.model_selection._split.StratifiedKFold',
        ],
    )

client = MlflowClient()
version = max(client.search_model_versions(f"name='{MODEL}'"), key=lambda item: int(item.version))
client.set_registered_model_alias(MODEL, 'Champion', version.version)
print(f'models:/{MODEL}@Champion')

2026/08/16 13:59:02 WARNING mlflow.utils.databricks_utils: Failed to create databricks SDK workspace client, error: ValueError('default auth: cannot configure default credentials, please check https://docs.databricks.com/en/dev-tools/auth.html#databricks-client-unified-authentication to configure credentials for your preferred authentication method.'). Falling back to legacy authentication.


MlflowException: Reading Databricks credential configuration failed with MLflow tracking URI 'databricks'. Please ensure that the 'databricks-sdk' PyPI library is installed, the tracking URI is set correctly, and Databricks authentication is properly configured. The tracking URI can be either 'databricks' (using profile name specified by 'DATABRICKS_CONFIG_PROFILE' environment variable or using 'DEFAULT' authentication profile if 'DATABRICKS_CONFIG_PROFILE' environment variable does not exist) or 'databricks://{profile}'. You can configure Databricks authentication in several ways, for example by specifying environment variables (e.g. DATABRICKS_HOST + DATABRICKS_TOKEN) or logging in using 'databricks auth login'. 
For details on configuring Databricks authentication, please refer to 'https://docs.databricks.com/en/dev-tools/auth/index.html#unified-auth'.